# Simple ReAct Agent from Scratch

In [1]:
# based on https://til.simonwillison.net/llms/python-react-pattern
# Adapted to use an Anthropic API key

In [33]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()  # Load environment variables from .env file
from openai import OpenAI

In [34]:
client = OpenAI()

In [35]:
chat_completion = client.chat.completions.create( # make an API call to the model
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Hello, how are you?"}] # passing conversation history
)

In [36]:
chat_completion.choices[0].message.content # extracting the content of the first message in the response back out

"Hello! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with whatever you need. How can I assist you today?"

In [32]:
class Agent:
    def __init__(self, system=""): # parameterized by a system message (personality, constraints, or instructions)
        self.system = system # saved as an attribute of the Agent instance
        self.messages = [] # keeping track of the conversation history in a list of messages
        if self.system:
            self.messages.append({"role": "system", "content": self.system}) # if a system message is provided, add it to the conversation history

    def __call__(self, message):
        """
        Take a message that comes in as a string and append it to the conversation history
        """
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result
    
    def execute(self):
        """
        Calling the OpenAI client
        """
        completion = client.chat.completions.create(
                        model="gpt-4o", 
                        temperature=0, # deterministic output
                        messages=self.messages) # pass in the list of messages that have been accumulated so far
        return completion.choices[0].message.content # return the content of the first message in the response back out

This is the basic agent completed, now I will move on to create a ReAct (Reasoning and Action) agent.

In [7]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

Here are the tools mentioned in the prompt:

In [8]:
def calculate(what):
    return eval(what)

def average_dog_weight(name): # mocked with returns
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = { # dictionary to map the name of the function to the function itself
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

These functions are a toy example and will get more specific to a problem later.

In [9]:
abot = Agent(prompt) #initialize the Agent with the prompt

In [10]:
# Call it once with an initial question
result = abot("How much does a toy poodle weigh?")
print(result)

Thought: I should look up the average weight of a Toy Poodle using average_dog_weight.
Action: average_dog_weight: Toy Poodle
PAUSE


In [11]:
# Following the logic
result = average_dog_weight("Toy Poodle")
result

'a toy poodles average weight is 7 lbs'

In [12]:
next_prompt = "Observation: {}".format(result) # formatting that into the next prompt

In [13]:
abot(next_prompt) # calling the Agent again with the next prompt

'Answer: A Toy Poodle weighs an average of 7 lbs.'

In [14]:
#looking at more details
abot.messages # looking at the messages that have been accumulated so far

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},
 {'role': 'user', 'content': 'How much does a 

In [15]:
abot = Agent(prompt) #re-initialize the Agent with the prompt

In [16]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
abot(question)

'Thought: I need to find the average weight of both a Border Collie and a Scottish Terrier, then add them together to find the combined weight.\nAction: average_dog_weight: Border Collie\nPAUSE'

In [17]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))
print(next_prompt)

Observation: a Border Collies average weight is 37 lbs


In [18]:
abot(next_prompt) # calling the Agent again with the next prompt

'Action: average_dog_weight: Scottish Terrier\nPAUSE'

In [19]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))
print(next_prompt)

Observation: Scottish Terriers average 20 lbs


In [20]:
abot(next_prompt)

'Action: calculate: 37 + 20\nPAUSE'

In [21]:
next_prompt = "Observation: {}".format(calculate("37 + 20"))
print(next_prompt)

Observation: 57


In [ ]:
abot(next_prompt) # final call to the Agent with the final observation

'Answer: The combined average weight of a Border Collie and a Scottish Terrier is 57 lbs.'

That is great but all manual. Next I will put it in a loop. 

In [23]:
# Parsing the LLM's response to determine if we will take an action, or if it is the final answer.
action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action

<>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
/tmp/ipykernel_18132/757013114.py:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
  action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action


In [30]:
def query(question, max_turns=5): # take in a question and run the process automatically. Max turns controls how long it runs for
    i = 0 # keeping track of how many iterations we have done so far
    bot = Agent(prompt) #intialize an Agent with a default prompt
    next_prompt = question 
    while i < max_turns: # define a query function
        i += 1
        result = bot(next_prompt) #calling the agent and getting a result back
        print(result) 
        actions = [ # parsing the responses with the regular expression to see what actions are available to run
            action_re.match(a) 
            for a in result.split('\n') 
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run (the output isn't an answer)
            action, action_input = actions[0].groups() # action: function to call, action_input: input to that function
            if action not in known_actions: 
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input) # looking up the action which we should take, and calling it on the action input. The result is saved to observation.
            print("Observation:", observation) # for debugging
            next_prompt = "Observation: {}".format(observation)
        else: # the output is an answer
            return

In [31]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
query(question)

Thought: To find the combined weight of the two dogs, I need to find the average weight of each breed using the average_dog_weight action and then add them together. I will first find the weight of a border collie.
Action: average_dog_weight: Border Collie
PAUSE
 -- running average_dog_weight Border Collie
Observation: a Border Collies average weight is 37 lbs
Thought: Next, I need to find the average weight of a Scottish Terrier.
Action: average_dog_weight: Scottish Terrier
PAUSE
 -- running average_dog_weight Scottish Terrier
Observation: Scottish Terriers average 20 lbs
Thought: Now that I have the average weights of both a Border Collie and a Scottish Terrier, I can calculate their combined weight by adding these two values together.
Action: calculate: 37 + 20
PAUSE
 -- running calculate 37 + 20
Observation: 57
Answer: The combined average weight of a Border Collie and a Scottish Terrier is 57 lbs.
